# Data Migration: SQL Server to Postgres

In [1]:
import os
import pandas as pd
import pyodbc
import psycopg2
from psycopg2.extras import execute_values
from dotenv import load_dotenv

In [30]:
os.getcwd()

'c:\\Users\\opeab\\OneDrive\\Documents\\Github\\sql-server-to-postgres-migration'

In [31]:
os.listdir()

['.env',
 '.git',
 '.gitignore',
 '.venv',
 'generate_data.py',
 'README.md',
 'run_migration_uat.ipynb']

In [32]:
os.path.isfile(".env")

True

## 1. Load credentials

In [2]:
load_dotenv(".env")

True

In [3]:
sql_host = os.getenv("SQL_SERVER_HOST")
sql_db = os.getenv("SQL_SERVER_DB")

In [4]:
print(f"SQL SERVER HOST: {sql_host}")
print(f"SQL SERVER DB: {sql_db}")

SQL SERVER HOST: OPSY\SQLEXPRESS
SQL SERVER DB: TransactionDB_UAT


In [5]:
pg_host = os.getenv("POSTGRES_HOST")
pg_port = os.getenv("POSTGRES_PORT")
pg_db = os.getenv("POSTGRES_DB")
pg_user = os.getenv("POSTGRES_USER")
pg_password = os.getenv("POSTGRES_PASSWORD")

In [6]:
print(f"POSTGRES HOST: {pg_host}")
print(f"POSTGRES PORT: {pg_port}")
print(f"POSTGRES DB: {pg_db}")
print(f"POSTGRES USER: {pg_user}")
print(f"POSTGRES PASSWORD: {pg_password}")

POSTGRES HOST: localhost
POSTGRES PORT: 5432
POSTGRES DB: transaction_uat
POSTGRES USER: postgres
POSTGRES PASSWORD: Positive*1


## 2. Connect to SQL Server

In [7]:
print("Connecting  to SQL Server...")
print(f"   Server: {sql_host}")
print(f"   Database: {sql_db}")

Connecting  to SQL Server...
   Server: OPSY\SQLEXPRESS
   Database: TransactionDB_UAT


In [8]:
try:
    sql_conn_string = (
        f"DRIVER={{ODBC Driver 17 for SQL Server}};"
        f"SERVER={sql_host};"
        f"DATABASE={sql_db};"
        "Trusted_Connection=yes;"
    )

    sql_conn = pyodbc.connect(sql_conn_string)
    sql_cursor = sql_conn.cursor()
    print("[SUCCESS] SQL Server connection established.")

except Exception as e:
    print(f"SQL Server connection failed: {e}")
    print(""" How to troubleshoot:
          > 1. Check server name in .env file is correct
          . 2. Verify SQL Server is running
          > 3. Check Windows Authentication is enabled
            ....
""")

[SUCCESS] SQL Server connection established.


# 3. Connect to PostgreSQL

In [9]:
print("Connecting to PostgreSQL...")
print(f"    Server: {pg_host}")
print(f"    Database: {pg_db}")

Connecting to PostgreSQL...
    Server: localhost
    Database: transaction_uat


In [10]:
try: 
    pg_conn = psycopg2.connect(
        host=pg_host,
        port=pg_port,
        database=pg_db,
        user=pg_user,
        password=pg_password
    )

    pg_cursor=pg_conn.cursor()
    pg_cursor.execute("SELECT version();")

    pg_version = pg_cursor.fetchone()[0]

    print("Connected to PostgreSQL successfully!")
    print(f"    Version: {pg_version[:50]}...\n")


except psycopg2.OperationalError as e:
    print(f"Postgres connection failed:{e}")
    print(""" How to troubleshoot:
          > 1. Check Postgres is running
          > 2. Verify username + password
          > 3. Check database exists
        ....

""")
    
except Exception as e:
    print(f" Unexpected error: {e}")
    raise

Connected to PostgreSQL successfully!
    Version: PostgreSQL 18.1 on x86_64-windows, compiled by msv...



# 4. Define the tables to migrate

### Migration order

- Categories (no dependencies)
- Supplies (no dependencies)
- Customers (no dependencies)
- Products (depends on Categories and Suppliers)


In [11]:
tables_to_migrate = ['Categories', 'Suppliers', 'Customers', 'Products']
print(tables_to_migrate)

['Categories', 'Suppliers', 'Customers', 'Products']


In [12]:
print("Table to migrate:")
for i, table in enumerate(tables_to_migrate, 1):
    print(f"    {i}. {table}")

total_no_tbls = len(tables_to_migrate)
print(f"\nTotal no of tables to migrate: {total_no_tbls}")

Table to migrate:
    1. Categories
    2. Suppliers
    3. Customers
    4. Products

Total no of tables to migrate: 4


# 5. Run pre-migration checks

In [13]:
print("=" * 50)
print(">>> Check 1: ROW COUNTS")
print("=" * 50)

>>> Check 1: ROW COUNTS


In [15]:
baseline_counts = {}


try:
    for table in tables_to_migrate:
        quoted_table = f"[{table}]"
        row_count_query = f"SELECT COUNT(*) as total_rows FROM {quoted_table};"
        sql_cursor.execute(row_count_query)
        count = sql_cursor.fetchone()[0]

        baseline_counts[table] = count
        print(f"{table:15} {count:>12} rows")

    total_rows = sum(baseline_counts.values())
    print(f"{'-' * 30}")
    print(f"{'TOTAL':15} {total_rows:>12} rows")
    print("\n Baseline captured! ")

except Exception as e:
    print(f"Failed to get baseline counts: {e}")
    raise

Categories                 8 rows
Suppliers               5000 rows
Customers             900000 rows
Products              150000 rows
------------------------------
TOTAL                1055008 rows

 Baseline captured! 


In [16]:
tables_to_migrate = {'Categories', 'Suppliers', 'Customers', 'Products'}
print(tables_to_migrate)

{'Suppliers', 'Categories', 'Customers', 'Products'}


In [18]:
print("Table to migrate:")
for i, table in enumerate(tables_to_migrate, 1):
    print(f"    {i}.  {table}")

print(f"\nTotal no of tables to migrate: {len(tables_to_migrate)}")

Table to migrate:
    1.  Suppliers
    2.  Categories
    3.  Customers
    4.  Products

Total no of tables to migrate: 4
